# Experiment 1 — the first rule against the benchmark

**Steps 4, 5 and 6 of 8 — portfolio, backtest, attribution — for one idea.**

**It produces** a book in `Portfolio/`, a track record in `Backtest/` and a breakdown of the
return in `Attribution/`. **It prevents** a good signal in a portfolio nobody could hold, paper
returns that real trading would have erased, and factor beta sold as alpha.

Experiment 1 is the **first rule tested against the benchmark** `BRAINSTORMING_1.md` names, and
the yardstick every later experiment is also measured against. It is not a null. It is a real
strategy with a real return, and it can graduate like any other — **its rules freeze once
`FINDINGS_1.md` reports**, because a change to them invalidates every comparison in `RESULTS.md`.
Improvements go into a new experiment, unless the owner rewrites this one, as `AGENTS.md` says.

The hypothesis is in [`BLUEPRINT_1.md`](BLUEPRINT_1.md), **written before this notebook's rule**.
The running log is [`JOURNAL_1.md`](JOURNAL_1.md); the results that survive are in
[`FINDINGS_1.md`](FINDINGS_1.md).

**This notebook is empty by design.** Each section below says what is expected in it. Write the
cells.

## Position in the pipeline

This notebook is **only the strategy**. The universe and the data are built by earlier stages and
are simply read here:

```
Universe/universe.ipynb   ->  Security_Master.csv
Data/curator.py           ->  Data/Curator/Time_Series/     m_* + c_*
Data/refinery.py          ->  Data/Refinery/Time_Series/    + r_*      <- this notebook reads here
Data/analyzer.ipynb       ->  the measurements the blueprint's predictions came from
        |
        v
experiment_1.ipynb        ->  Portfolio/  ->  Backtest/  ->  Attribution/
```

**What this notebook does not do.** It does not download anything, profile the universe, or compute
a signal. A number about the data itself belongs in the Universe or Data stage — that separation is
what keeps every experiment comparable, because all of them read the identical panel.

**Four modules beside this notebook are shared by every experiment** — `securities_panel.py`,
`portfolio_construction.py`, `backtest_engine.py` and `attribution_analysis.py`, one per Lab
library — and no strategy column is named in any of them. The benchmark loads the panel through
`securities_panel.py` like every later experiment, so the comparison is on the rule and nothing
else.

## The section contract

Every experiment notebook has the same shape, so anyone who has read one can read all of them.
**Everything below section 2 is strategy-agnostic**, given the three objects that section produces.

| Section | Contains |
| --- | --- |
| 0 · Setup | paths, and **the strategy's columns** — the only strategy names in this notebook outside the rule |
| 1 · The panel | load the refined files and reshape them |
| 2 · The rule | selection, sizing, timing. **The one cell you write** |
| 2.1 · Invariants | what every rule must pass, whatever it is |
| 3 · Construction | the book, and the diagnostics a person would run it on |
| 4 · Backtest | one engine pass, guarded import, reports-and-skips without a licence |
| 5 · Attribution | where the return came from, guarded the same way |
| 6 · Counterfactuals | the arms that price who earned the idiosyncratic share |
| 7 · Verdict | what it concluded, **in words** |
| Handoff | what the next stage consumes, and what this one left open |
| 8 · Verify | assertions that raise when the output is wrong: the invariants, the weight file read back, the days each engine run valued against its window's trading days |

## 0 · Setup

Paths, and the strategy's columns. **Declare them here, not in a shared module**, so a signal never
becomes every later experiment's default without anyone deciding it. Read only the columns the
strategy consumes.

Three price columns do **three different jobs**, and getting them out of step is silent — the
backtest P&L and the attribution would quietly run on different bases:

| Role | Basis | Why |
| --- | --- | --- |
| Daily mark | dividend-and-split adjusted close | total-return valuation between rebalances |
| Fill | dividend-and-split adjusted VWAP | the price a trade actually gets |
| Commission | **unadjusted** VWAP | per-share cents ride on the unadjusted share count |

A provider may return its VWAP columns as null, which is why the Curator reconstructs both VWAPs as
`c_*`, and nothing below reads a provider's own VWAP column, whichever provider `Data/curator.py`
asks for.

In [ ]:
# EXAMPLE-ONLY CELL
import os
import pathlib
import sys

import pandas

NOTEBOOK_DIRECTORY = pathlib.Path.cwd()
REPOSITORY_ROOT = next(
    parent
    for parent in (NOTEBOOK_DIRECTORY, *NOTEBOOK_DIRECTORY.parents)
    if (parent / "Universe").is_dir()
)
os.chdir(REPOSITORY_ROOT)
sys.path.insert(0, str(REPOSITORY_ROOT / "Experiments"))
sys.path.insert(0, str(REPOSITORY_ROOT / "Data"))

import attribution_analysis  # noqa: E402 - the paths above have to exist first
import backtest_engine  # noqa: E402
import hand_supplied  # noqa: E402
import portfolio_construction  # noqa: E402
import securities_panel  # noqa: E402

EXPERIMENT_DIRECTORY = REPOSITORY_ROOT / "Experiments" / "Experiment_1"

# The strategy's columns, named here and nowhere else. Three prices do three different jobs.
SIGNAL_COLUMN = "r_trend_50_200"
RANKING_COLUMN = "r_liquidity_rank"
MARK_COLUMN = "m_close_dividend_and_split_adjusted"
FILL_COLUMN = "c_vwap_dividend_and_split_adjusted"

# The rule's settings, from BLUEPRINT_1.md of 2026-09-23, committed before this cell was written.
BOOK_SIZE = 20
BUFFER_RANK = 30
REEQUALISE = "month"
AVERAGES = (50, 200)
# Excluded by name: no price file at all, or an adjusted price that multiplies by more than six in
# a day, which is a bad print rather than a return. All five are blocking rows in Data_Issues.csv.
EXCLUDED_IDENTIFIERS = (
    "CIT",
    "FMC",
    "LCI",
    "MIC",
    "PARA",
)
POINT_IN_TIME_START = pandas.Timestamp("2017-01-03")
# Everything after this date is held out: nothing below reads it.
WINDOW_END = pandas.Timestamp("2026-06-01")
# The kill switch's three sub-periods, fixed in the blueprint.
SUB_PERIODS = (
    (pandas.Timestamp("2017-01-03"), pandas.Timestamp("2019-12-31")),
    (pandas.Timestamp("2020-01-02"), pandas.Timestamp("2022-12-30")),
    (pandas.Timestamp("2023-01-03"), pandas.Timestamp("2026-06-01")),
)
# The diagnostic arm: the first design's code at a band that keeps its behaviour at twenty names --
# one swapped name is a tenth and waits, two are a fifth and trade.
DIAGNOSTIC_BAND = 0.15
# The margins over the control that prediction 1 and the kill switch require, at the headline costs.
SHARPE_MARGIN = 0.03
CAGR_MARGIN_POINTS = 0.5
# Criterion 3: the rule's Sharpe margin over its control keeps its sign in at least this many cells.
CELLS_REQUIRED = 12
TRADED_VALUE_COLUMN = "c_daily_traded_value"
# The perturbation, one setting at a time around the rule, as the blueprint fixed it.
SWEEP = (
    ("book size 10", {"book_size": 10, "buffer_rank": 15}),
    ("book size 15", {"book_size": 15, "buffer_rank": 23}),
    ("book size 25", {"book_size": 25, "buffer_rank": 38}),
    ("book size 30", {"book_size": 30, "buffer_rank": 45}),
    ("buffer 20, none", {"buffer_rank": 20}),
    ("buffer 25", {"buffer_rank": 25}),
    ("buffer 40", {"buffer_rank": 40}),
    ("re-equalise never", {"reequalise": "never"}),
    ("re-equalise quarterly", {"reequalise": "quarter"}),
    ("averages 40 and 160", {"averages": (40, 160)}),
    ("averages 60 and 250", {"averages": (60, 250)}),
    ("ranking window 21 days", {"ranking_window": 21}),
    ("ranking window 126 days", {"ranking_window": 126}),
    ("acted on 5 days late", {"delay": 5}),
    ("acted on 21 days late", {"delay": 21}),
)

print(f"repository root: {REPOSITORY_ROOT}")
print(f"engine installed: {backtest_engine.ENGINE_INSTALLED}")
print(f"attribution installed: {attribution_analysis.LIBRARY_INSTALLED}")

## 1 · The panel

Load the refined files, resolve **one position per security**, and reshape to matrices.

A point-in-time universe contains renamed securities: two identifiers sharing one identity, each
carrying part of the history. Left alone they are two independent positions and the book
double-counts at the changeover. Key positions by a stable identity — an ISIN where the seed
carries one — falling back to the identifier itself, and where two legs overlap on a date let the
leg still reporting later win.

The long panel then becomes one wide `dates x securities` matrix per input, which is what makes the
whole rule in section 2 a handful of vectorised lines instead of a loop over files.

In [ ]:
# EXAMPLE-ONLY CELL
matrices = securities_panel.load_matrices((
    SIGNAL_COLUMN,
    RANKING_COLUMN,
    MARK_COLUMN,
    FILL_COLUMN,
    TRADED_VALUE_COLUMN,
))
# Each column comes back on the dates it has values; every matrix is put on the price calendar, and
# nothing after the window's end is kept, so the held-out months cannot reach a single cell below.
calendar = matrices[MARK_COLUMN].index
in_panel = calendar <= WINDOW_END
mark = matrices[MARK_COLUMN].loc[in_panel]
signal = matrices[SIGNAL_COLUMN].reindex(index=mark.index, columns=mark.columns)
ranking = matrices[RANKING_COLUMN].reindex(index=mark.index, columns=mark.columns)
fill = matrices[FILL_COLUMN].reindex(index=mark.index, columns=mark.columns)
traded_value = matrices[TRADED_VALUE_COLUMN].reindex(index=mark.index, columns=mark.columns)
returns = mark.pct_change(fill_method=None)

# Membership, point in time: the index's own daily holdings, read as the desk ships them and mapped
# onto positions rather than listings, because a company that changed ticker is one position here.
position_keys = securities_panel.read_position_keys()
holdings = hand_supplied.read_benchmark_holdings()
holdings.columns = [position_keys.get(column, column) for column in holdings.columns]
in_index = (holdings > 0).T.groupby(level=0).any().T
membership = in_index.reindex(
    index=mark.index,
    columns=mark.columns,
).ffill()
membership = membership.where(membership.notna(), False).astype(bool)

# The cash proxy's own return, so the slot book's cash drifts the way the engine will price it.
cash_file = backtest_engine.MARKET_DATA_DIRECTORY / f"{backtest_engine.CASH_IDENTIFIER}.csv"
cash_prices = pandas.read_csv(
    cash_file,
    usecols=["m_date", MARK_COLUMN],
    parse_dates=["m_date"],
    index_col="m_date",
)[MARK_COLUMN]
cash_returns = cash_prices.reindex(mark.index).pct_change(fill_method=None).fillna(0.0)

print(f"panel: {mark.shape[0]} dates x {mark.shape[1]} positions, to {mark.index.max().date()}")
print(f"index membership known from {holdings.index.min().date()} to {holdings.index.max().date()}")

## 2 · The rule — the one cell you write

Three statements, in order: **who is eligible**, **how much of each**, and **when to trade**.

**The contract this cell must satisfy** — everything below reads exactly these three objects:

| Object | Type | Meaning |
| --- | --- | --- |
| `selected_matrix` | `dates x securities` boolean | what the book holds on each day |
| `REBALANCE_DATES` | a date index | the days the book is re-struck |
| `target_weights` | `REBALANCE_DATES x securities` float, rows summing to **at most** 1.0 | the book on each of those days |

**Rows sum to at most one, not to exactly one.** A book that must be fully invested cannot express
a defensive strategy. The residual becomes cash in section 3.1, parked in a real priced instrument,
because the engine's weight file has no cash row of its own.

**Sizing is a seam, not a decision buried in the rule.** Hand the eligible set and a returns
history that has already been cut off before today to a weighting function in
`portfolio_construction.py`, and swapping equal weight for inverse volatility, hierarchical risk
parity or any other method of the KaxaNuk Portfolio Construction library is one line — the module
builds the library's method on that cut history, one rebalance date at a time. That is what makes
two experiments comparable rather than merely adjacent.

**Trade only when something changed.** A signal that has not moved is not a reason to pay
commission.

### Two look-aheads, both stated plainly

**The lag.** The eligible set used on rebalance date *t* is the one observed at *t-1*, and the fill
happens at *t*'s price — a full day between the signal and the fill.

**The delisting exit.** A security that delists must be sold on the **last day it still has a fill
price**, and knowing that day is its last requires seeing the next one. This is the standard
backtest compromise — the alternative, carrying a position that can never be exited, is a larger
distortion — and it is implemented by making a name ineligible on that final day, so the set
changes, the rebalance fires, and the position is sold while a price still exists.

In [ ]:
# EXAMPLE-ONLY CELL
# The rule, from BLUEPRINT_1.md: who may be held, who enters, when to sell, and how much of each.
# Written after the blueprint was committed, which the history shows.
excluded = [identifier for identifier in EXCLUDED_IDENTIFIERS if identifier in mark.columns]
tradable = mark.notna() & fill.notna()


def trend_state(averages):
    """
    The cross as a state, 1 when the shorter average is above the longer.

    The rule's own pair reads the refinery's column; a perturbed pair is computed here the same
    way, on the same adjusted close, because the refinery carries one pair.
    """
    if tuple(averages) == (50, 200):
        return signal > 0

    shorter = mark.rolling(averages[0]).mean()
    longer = mark.rolling(averages[1]).mean()

    return shorter > longer


def ranking_matrix(ranking_window):
    """
    The per-date percentile of average traded value; the rule's own window reads the refinery's
    column, and a perturbed window is computed here the same way.
    """
    if ranking_window == 63:
        return ranking

    return traded_value.rolling(ranking_window).mean().rank(axis=1, pct=True)


def eligible_matrix(use_cross, averages):
    """Members with a price and a fill price, and the cross at 1 when the rule uses it."""
    state = trend_state(averages) if use_cross else tradable
    eligible = state & tradable & membership
    kept = eligible.drop(columns=excluded, errors="ignore")

    return kept.reindex(columns=mark.columns, fill_value=False).astype(bool)


def build_book(
    start,
    end,
    book_size=BOOK_SIZE,
    buffer_rank=BUFFER_RANK,
    reequalise=REEQUALISE,
    averages=AVERAGES,
    use_cross=True,
    decision_dates=None,
    ranking_window=63,
    delay=0,
):
    """
    The rule over a window: sell the day after a name stops being eligible, fill the slot from the
    top of the ranking, and check the buffer and re-equalise on the first trading day of each month.

    `use_cross=False` is the control: the same names, the same exits for leaving the index or losing
    a price, the same buffer and re-equalisation, trading only on the rule's `decision_dates`.
    `delay` acts on the whole rule that many trading days later than the one-day lag.
    """
    in_window = (mark.index >= start) & (mark.index <= end)
    eligible = eligible_matrix(use_cross, averages).shift(delay, fill_value=False)
    lagged = portfolio_construction.lag_eligibility(eligible.loc[in_window])
    lagged_ranking = ranking_matrix(ranking_window).shift(1 + delay).loc[in_window]
    settings = portfolio_construction.SlotSettings(
        book_size=book_size,
        buffer_rank=buffer_rank,
        reequalise_dates=portfolio_construction.first_trading_days(lagged.index, reequalise),
        decision_dates=decision_dates,
    )

    return portfolio_construction.build_slot_book(
        lagged,
        lagged_ranking,
        returns.loc[in_window],
        cash_returns.loc[in_window],
        settings,
    )


def build_first_design_book(start, end, book_size=BOOK_SIZE):
    """
    The diagnostic arm: twenty names with the first design's timing -- the whole set re-equalised
    whenever it moves by a tenth -- in place of the immediate exit and the buffer.
    """
    in_window = (mark.index >= start) & (mark.index <= end)
    eligible = eligible_matrix(True, AVERAGES)
    standing = ranking.where(eligible)
    held = standing.rank(axis=1, ascending=False, method="first") <= book_size
    selected = (held & eligible).loc[in_window]
    lagged = portfolio_construction.lag_eligibility(selected)
    dates = portfolio_construction.select_rebalance_dates(lagged, DIAGNOSTIC_BAND)

    return portfolio_construction.build_weights(
        lagged,
        returns.loc[mark.index <= end],
        dates,
        "equal_weight",
        1.0 / book_size,
        1,
    )


target_weights = build_book(POINT_IN_TIME_START, WINDOW_END)
TRADE_DATES = target_weights.index
control_weights = build_book(
    POINT_IN_TIME_START,
    WINDOW_END,
    use_cross=False,
    decision_dates=TRADE_DATES,
)
diagnostic_weights = build_first_design_book(POINT_IN_TIME_START, WINDOW_END)


def broken_holding(weights):
    """
    How long a book holds a name after its cross breaks: held name-days with the cross at 0 at the
    prior close, and the median run of such days. Read from the book's targets held forward, so it
    is the book's construction, not its return.
    """
    days = mark.index[(mark.index >= POINT_IN_TIME_START) & (mark.index <= WINDOW_END)]
    held = weights.reindex(days).ffill().fillna(0.0).gt(0)
    broken = held & ~(signal > 0).shift(1, fill_value=False).reindex_like(held)
    runs_of_days = []

    for name in broken.columns:
        flags = broken[name].to_numpy()
        run_length = 0

        for flag in flags:
            if flag:
                run_length += 1
            elif run_length > 0:
                runs_of_days.append(run_length)
                run_length = 0

        if run_length > 0:
            runs_of_days.append(run_length)

    lengths = pandas.Series(runs_of_days, dtype=float)

    return {
        "held name-days": int(held.sum().sum()),
        "broken name-days": int(broken.sum().sum()),
        "median days held after a break": float(lengths.median()) if len(lengths) else 0.0,
    }


REEQUALISE_DATES = portfolio_construction.first_trading_days(
    mark.index[(mark.index >= POINT_IN_TIME_START) & (mark.index <= WINDOW_END)],
    REEQUALISE,
)
print(f"the rule: {len(TRADE_DATES)} trade dates, {TRADE_DATES.min().date()} to "
      f"{TRADE_DATES.max().date()}")
print(f"first trading days of a month in the window: {len(REEQUALISE_DATES)}, from "
      f"{REEQUALISE_DATES.min().date()}")
print(f"the control: {len(control_weights)} trade dates, all of them the rule's: "
      f"{set(control_weights.index) <= set(TRADE_DATES)}")
print(f"the diagnostic arm: {len(diagnostic_weights)} rebalances")
rule_delay = broken_holding(target_weights)
arm_delay = broken_holding(diagnostic_weights)
ARM_REPRODUCES_DELAY = arm_delay["median days held after a break"] > 1
print(f"broken holding, the rule: {rule_delay}")
print(f"broken holding, the diagnostic arm: {arm_delay}")
print(f"the arm reproduces the first design's delay: {ARM_REPRODUCES_DELAY}")

## 2.1 · Invariants

Cheap to check here, expensive to discover inside a P&L. **Every rule must pass these unchanged**,
whatever the strategy is:

- no book is more than fully invested, and none is negatively invested;
- no negative weights, if the strategy is long-only;
- **every security paid for today had its signal on at the prior close** — check the signal itself,
  not the composed eligibility, because the signal is the thing that had to exist in advance;
- every security bought is tradable on the day it is bought, so a fill price exists;
- nothing is still held on a day after it stopped being tradable.

In [ ]:
# EXAMPLE-ONLY CELL
# Cheap here, expensive inside a P&L. Every rule must pass these unchanged.
invested = target_weights.sum(axis=1)
bought = target_weights > 0
state_yesterday = (signal > 0).shift(1, fill_value=False).reindex_like(bought)
member_yesterday = membership.shift(1, fill_value=False).reindex_like(bought)
tradable_yesterday = tradable.shift(1, fill_value=False).reindex_like(bought)
checks = {
    "no book more than fully invested": bool((invested <= 1.0 + 1e-9).all()),
    "no book negatively invested": bool((invested >= -1e-9).all()),
    "no negative weights": bool((target_weights >= -1e-9).all().all()),
    "no more names than the book holds": bool((bought.sum(axis=1) <= BOOK_SIZE).all()),
    # The signal itself, not the composed eligibility: the signal is what had to exist in advance.
    "every holding had its cross at 1 at the prior close": bool(
        (~bought | state_yesterday).all().all()
    ),
    "every holding was in the index at the prior close": bool(
        (~bought | member_yesterday).all().all()
    ),
    "every holding was tradable at the prior close": bool(
        (~bought | tradable_yesterday).all().all()
    ),
    "no excluded name is ever held": bool(not bought[excluded].any().any()),
    "the book trades on fewer days than it does not": bool(
        len(TRADE_DATES)
        < ((mark.index >= POINT_IN_TIME_START) & (mark.index <= WINDOW_END)).sum() / 2
    ),
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")

## 3 · Construction — is this a book you would actually run?

**This is where step 4, Portfolio Construction, lives.** Four properties, each with a failure mode
a performance chart would hide:

| Property | What a bad value would mean |
| --- | --- |
| Invested share over time | the eligibility column is not doing what the analyzer says it does |
| Trigger frequency and turnover | the rule fires so often that this is a transaction-cost question, not an alpha one |
| Holdings and concentration | a "diversified" label on a book that is one or two positions |
| Group drift | the strategy is a disguised bet on one group rather than a rotation between them |

Measure turnover **target-to-target**. The realised figure is lower, because between rebalances the
winners drift up on their own; that calculation needs drifted weights and belongs to the backtest.

> **The blueprint's predictions about the *shape* of the book, rather than about its return, are
> settled here — before any backtest.** They are the first ones that can be wrong, and the cheapest
> to be wrong about.

## 3.1 · Write the deliverables

Two views of the same book, because two readers need it: a **long, human-readable** one with names
and classifications attached, and a **wide, identifier-keyed** `portfolio_weights.csv` — the
backtest engine's input, which looks each identifier up in the market-data folder and so has to
speak in identifiers, not in stitched positions.

**This is where cash becomes a position.** Everything above lets a book be less than fully
invested; here the residual becomes a weight in the cash proxy, so the engine charges commission on
going to cash and earns the yield while there. A strategy whose defining move is *sell everything*
has to pay for it.

In [ ]:
# EXAMPLE-ONLY CELL
# Is this a book you would actually run? The target weights, held forward between trade dates --
# drift is the engine's, and this is the book's shape, not its return.
window_days = mark.index[(mark.index >= POINT_IN_TIME_START) & (mark.index <= WINDOW_END)]
years = (WINDOW_END - POINT_IN_TIME_START).days / 365.25
daily_book = target_weights.reindex(window_days).ffill().fillna(0.0)
invested_share = daily_book.sum(axis=1)
holdings_count = daily_book.gt(0).sum(axis=1)
effective_positions = 1 / daily_book.pow(2).sum(axis=1).replace(0, pandas.NA)
turnover = target_weights.diff().abs().sum(axis=1) / 2
exits = (target_weights.shift(1).fillna(0.0).gt(0) & target_weights.eq(0)).sum(axis=1)
entries = (target_weights.shift(1).fillna(0.0).eq(0) & target_weights.gt(0)).sum(axis=1)

# A round trip: a name sold and bought back within five trading days -- the cross flickering.
position_of_day = pandas.Series(range(len(window_days)), index=window_days)
round_trips = 0

for name in target_weights.columns:
    held_path = target_weights[name].gt(0)
    sold_on = held_path.index[held_path.shift(1, fill_value=False) & ~held_path]
    bought_on = held_path.index[~held_path.shift(1, fill_value=False) & held_path]

    for sale in sold_on:
        later = bought_on[bought_on > sale]

        if len(later) > 0 and position_of_day[later[0]] - position_of_day[sale] <= 5:
            round_trips += 1

# Held on a day the cross closed at 0: the rule sells the next day, so each break costs one such
# day and no more. The lagged measure, `broken_holding` in section 2, is zero by construction.
broken_held = (daily_book.gt(0) & ~(signal > 0).reindex_like(daily_book).fillna(False)).sum().sum()

construction = pandas.Series({
    "trade dates": len(TRADE_DATES),
    "trade dates per year": round(len(TRADE_DATES) / years, 1),
    "exits": int(exits.sum()),
    "entries": int(entries.sum()),
    "round trips within five days": round_trips,
    "mean one-way turnover per trade date": round(turnover.mean(), 3),
    "annual turnover, target to target": round(turnover.sum() / years, 2),
    "mean invested share": round(invested_share.mean(), 3),
    "lowest invested share": round(invested_share.min(), 3),
    "mean holdings": round(holdings_count.mean(), 1),
    "mean effective positions": round(float(effective_positions.mean()), 1),
    "held name-days on a day the cross closed at 0": int(broken_held),
    "held name-days": int(daily_book.gt(0).sum().sum()),
})
print(construction.to_string())

# Capacity, criterion 4: every trade's size against the name's 63-day average traded value that day.
# The largest book the rule can run while no trade takes more than a given share of a day's
# volume is that share times the average traded value, over the weight traded, at the worst trade.
traded_weight = target_weights.diff().abs()
traded_weight.iloc[0] = target_weights.iloc[0]
average_traded_value = traded_value.rolling(63).mean().reindex(index=target_weights.index)
room = average_traded_value.reindex(columns=target_weights.columns) / traded_weight.where(
    traded_weight > 0
)
capacity_rows = {}

for participation in (0.01, 0.05):
    trade_capacity = (room * participation).stack().dropna()
    capacity_rows[f"{participation:.0%} of a day's traded value"] = {
        "worst trade, book size in dollars": float(trade_capacity.min()),
        "1st percentile of trades": float(trade_capacity.quantile(0.01)),
        "median trade": float(trade_capacity.median()),
    }

capacity = pandas.DataFrame(capacity_rows).T
print()
print("capacity -- the largest book the rule can trade at a share of each name's traded value:")
print(capacity.map(lambda value: f"{value:,.0f}").to_string())

master = pandas.read_csv("Universe/Security_Master.csv").set_index("main_identifier")
sector_of = master["sector"].reindex(daily_book.columns).fillna("unknown")
by_sector = daily_book.T.groupby(sector_of).sum().T
yearly_sector = by_sector.groupby(by_sector.index.year).mean()
print()
print("average weight by sector, by year (top 6 sectors):")
top_sectors = yearly_sector.mean().sort_values(ascending=False).head(6).index
print(yearly_sector[top_sectors].round(3).to_string())

In [ ]:
# EXAMPLE-ONLY CELL
# Two views of the same book. The long one is for a person; the wide one is the engine's input and
# has to speak in identifiers, because that is what its market-data folder is named by.
readable = target_weights.stack()
readable = readable[readable > 0].rename("weight").reset_index()
readable.columns = ["trade_date", "position", "weight"]
readable["name"] = readable["position"].map(master["name"])
readable["sector"] = readable["position"].map(master["sector"])
readable.to_csv(EXPERIMENT_DIRECTORY / "Portfolio" / "holdings_readable.csv", index=False)

by_identifier = securities_panel.expand_to_identifiers(target_weights)
weight_file = backtest_engine.write_weight_file(by_identifier, EXPERIMENT_DIRECTORY)
written = pandas.read_csv(weight_file, index_col=0)
print(f"{weight_file.name}: {written.shape[0]} identifiers x {written.shape[1]} trade dates")
print(f"every column sums to 1: {bool((written.sum(axis=0).sub(1.0).abs() < 1e-6).all())}")
cash_row = written.loc[backtest_engine.CASH_IDENTIFIER]
print(f"cash weight: first {cash_row.iloc[0]:.3f}, mean {cash_row.mean():.3f}")

## 4 · Backtest — KaxaNuk Backtest Engine

**In plain words:** run the rules over history, with costs, without peeking ahead.

The weight file goes to the licensed engine, which simulates the book share by share: it fills at a
real price, charges per-share commission on the unadjusted price, holds integer share counts and a
cash reserve, marks the portfolio daily between rebalances, and compares against the benchmarks.

**This is the only backtest in the repository**, and results are accepted **net** or not at all.
Clip the window to the shortest benchmark up front rather than discovering it as a crash, and report
which benchmark bound it.

> **Guard the import.** The engine installs from KaxaNuk's licensed index rather than PyPI, so this
> section reports what is missing and skips without it. Everything in `Portfolio/` is already
> written and does not depend on the engine — a clone with no licence gets a real book and no
> numbers, by design.

In [ ]:
# EXAMPLE-ONLY CELL
# Costs, the first design's, stated in BLUEPRINT_1.md rather than defaulted. The engine charges the
# commission setting in dollars a share, so 0.1 is ten cents on each share it trades; the first
# design's findings put its average at about eight cents, which this design did not re-measure.
# The realistic row beside it changes nothing else. The cash reserve is not a strategy choice:
# weights summing to exactly one leave nothing to pay commission with.
INITIAL_CAPITAL = 1_000_000
COMMISSION_CENTS = 0.1
REALISTIC_COMMISSION_CENTS = 0.005
SLIPPAGE_BASIS_POINTS = 5.0
CASH_RESERVE = 0.02
runs = {}


def price(label, weights, start, end, commission=COMMISSION_CENTS):
    """Write one book's weight file, run the engine over its window, and keep the result."""
    name = "portfolio_weights_" + label.replace(" ", "_").replace(",", "").replace("-", "_")
    by_identifier = securities_panel.expand_to_identifiers(weights)
    backtest_engine.write_weight_file(by_identifier, EXPERIMENT_DIRECTORY, name)
    configuration = backtest_engine.build_configuration(
        EXPERIMENT_DIRECTORY,
        start.date(),
        end.date(),
        INITIAL_CAPITAL,
        commission,
        SLIPPAGE_BASIS_POINTS,
        CASH_RESERVE,
        name,
    )
    result = backtest_engine.run_backtest(EXPERIMENT_DIRECTORY, configuration)

    if not result.success:
        message = f"the engine refused {label}: {result.error}"

        raise RuntimeError(message)

    runs[label] = {
        "result": result,
        "start": start,
        "end": end,
        "trade dates": len(weights),
        "window": backtest_engine.describe_window(result, end.date()),
    }
    statistics = result.data["portfolio_stats"]
    print(f"{label}: CAGR {statistics['Annualized Return (CAGR)']:.4f}, "
          f"Sharpe {statistics['Portfolio Sharpe Ratio']:.3f}, {runs[label]['window']}")


if not backtest_engine.ENGINE_INSTALLED:
    print("step 5 skipped: the KaxaNuk Backtest Engine is not installed")
else:
    price("the rule", target_weights, POINT_IN_TIME_START, WINDOW_END)
    price("the control", control_weights, POINT_IN_TIME_START, WINDOW_END)
    price("the diagnostic arm", diagnostic_weights, POINT_IN_TIME_START, WINDOW_END)
    price(
        "the rule, realistic costs",
        target_weights,
        POINT_IN_TIME_START,
        WINDOW_END,
        REALISTIC_COMMISSION_CENTS,
    )

    # The kill switch: each sub-period priced as a window of its own, the book entering on its
    # second trading day, the first the one-day lag allows, and its control held to that book's
    # own trade dates.
    for number, (sub_start, sub_end) in enumerate(SUB_PERIODS, start=1):
        sub_book = build_book(sub_start, sub_end)
        sub_control = build_book(sub_start, sub_end, use_cross=False, decision_dates=sub_book.index)
        price(f"sub-period {number}, the rule", sub_book, sub_start, sub_end)
        price(f"sub-period {number}, the control", sub_control, sub_start, sub_end)

    # The perturbation: one setting at a time, each cell beside its own control.
    for label, changes in SWEEP:
        cell_book = build_book(POINT_IN_TIME_START, WINDOW_END, **changes)
        control_changes = {
            key: value
            for key, value in changes.items()
            if key != "averages"
        }
        cell_control = build_book(
            POINT_IN_TIME_START,
            WINDOW_END,
            use_cross=False,
            decision_dates=cell_book.index,
            **control_changes,
        )
        price(f"{label}, the rule", cell_book, POINT_IN_TIME_START, WINDOW_END)
        price(f"{label}, the control", cell_control, POINT_IN_TIME_START, WINDOW_END)

In [ ]:
# EXAMPLE-ONLY CELL
# Every figure below comes from the engine. There is no second simulator in this repository.
if len(runs) == 0:
    print("step 5 skipped: no backtest to summarise")
else:
    rows = {}

    for label, run in runs.items():
        statistics = run["result"].data["portfolio_stats"]
        rows[label] = {
            "CAGR": statistics["Annualized Return (CAGR)"],
            "volatility": statistics["Annualized Volatility"],
            "Sharpe": statistics["Portfolio Sharpe Ratio"],
            "max drawdown": statistics["Max Drawdown"],
            "alpha vs index": statistics.get("Alpha"),
            "information ratio": statistics.get("Information Ratio"),
            "commissions": statistics["Total Commissions"],
            "slippage": statistics["Total Slippage Costs"],
            "trade dates": run["trade dates"],
        }

    benchmark_statistics = runs["the rule"]["result"].data["benchmark_stats"]
    rows["the index, same window"] = {
        "CAGR": benchmark_statistics["Annualized Return (CAGR)"],
        "volatility": benchmark_statistics["Annualized Volatility"],
        "Sharpe": benchmark_statistics["Portfolio Sharpe Ratio"],
        "max drawdown": benchmark_statistics["Max Drawdown"],
    }
    summary = pandas.DataFrame(rows).T
    headline_rows = [
        "the rule",
        "the control",
        "the diagnostic arm",
        "the rule, realistic costs",
        "the index, same window",
    ]
    print(summary.loc[headline_rows].round(4).to_string())
    print()

    # Prediction 1 and the success criterion: the rule against its control, by the fixed margins.
    sharpe_margin = rows["the rule"]["Sharpe"] - rows["the control"]["Sharpe"]
    cagr_margin = (rows["the rule"]["CAGR"] - rows["the control"]["CAGR"]) * 100
    MARGINS_MET = sharpe_margin >= SHARPE_MARGIN and cagr_margin >= CAGR_MARGIN_POINTS
    print(f"the rule minus its control: Sharpe {sharpe_margin:+.3f}, "
          f"CAGR {cagr_margin:+.2f} points")
    print(f"the margins of prediction 1 met: {MARGINS_MET}")

    # Betas, side by side, from the engine's own daily returns and its benchmark series.
    def engine_beta(result):
        """The book's beta to the engine's benchmark, from the two daily series it returned."""
        book_returns = result.data["Register_df"]["Returns"]
        benchmark_table = result.data["benchmark"].to_pandas()
        benchmark_returns = benchmark_table.set_index(
            pandas.to_datetime(benchmark_table["date_column"])
        )["daily_return"].astype(float)
        paired = pandas.concat([book_returns, benchmark_returns], axis=1, join="inner").dropna()

        return paired.iloc[:, 0].cov(paired.iloc[:, 1]) / paired.iloc[:, 1].var()

    for label in ("the rule", "the control", "the diagnostic arm"):
        print(f"beta to the index, {label}: {engine_beta(runs[label]['result']):.3f}")

    # The kill switch: two of three sub-periods.
    sub_rows = []

    for number in range(1, len(SUB_PERIODS) + 1):
        rule_row = rows[f"sub-period {number}, the rule"]
        control_row = rows[f"sub-period {number}, the control"]
        sub_rows.append({
            "sub-period": number,
            "Sharpe, the rule": rule_row["Sharpe"],
            "Sharpe, the control": control_row["Sharpe"],
            "CAGR, the rule": rule_row["CAGR"],
            "CAGR, the control": control_row["CAGR"],
            "the rule ahead on both": (
                rule_row["Sharpe"] > control_row["Sharpe"]
                and rule_row["CAGR"] > control_row["CAGR"]
            ),
        })

    kill_switch = pandas.DataFrame(sub_rows).set_index("sub-period")
    print()
    print(kill_switch.round(3).to_string())
    SUB_PERIODS_AHEAD = int(kill_switch["the rule ahead on both"].sum())
    KILL_SWITCH_TRIPS = (not MARGINS_MET) or SUB_PERIODS_AHEAD < 2
    print(f"sub-periods with the rule ahead of its control on both: {SUB_PERIODS_AHEAD} of 3")
    print(f"the kill switch trips: {KILL_SWITCH_TRIPS}")

    # The perturbation, read as curves: each cell's margin over its own control.
    sweep_rows = []

    for label, _ in SWEEP:
        rule_row = rows[f"{label}, the rule"]
        control_row = rows[f"{label}, the control"]
        sweep_rows.append({
            "cell": label,
            "Sharpe": rule_row["Sharpe"],
            "Sharpe over control": rule_row["Sharpe"] - control_row["Sharpe"],
            "CAGR over control, points": (rule_row["CAGR"] - control_row["CAGR"]) * 100,
            "trade dates": rule_row["trade dates"],
        })

    sweep_table = pandas.DataFrame(sweep_rows).set_index("cell")
    headline_sign = 1 if sharpe_margin > 0 else -1
    same_sign = int(((sweep_table["Sharpe over control"] > 0) == (headline_sign > 0)).sum())
    CRITERION_3_READ_AS_PASSED = same_sign >= CELLS_REQUIRED
    print()
    print(sweep_table.round(3).to_string())
    print(f"cells whose Sharpe margin keeps the rule's sign: {same_sign} of {len(SWEEP)}; "
          f"criterion 3 read as passed: {CRITERION_3_READ_AS_PASSED}")

## 5 · Attribution — KaxaNuk Attribution Analysis

**In plain words:** which part of the return did you actually earn?

The backtest says *how much* the book made; attribution says **where it came from**:

- **Brinson-Fachler**, the first cut: active return into an **allocation** effect — being
  overweight the right groups — and a **selection** effect, picking the right securities inside
  them. The exact lever that moved.
- **A factor model**, the second layer: excess return into **compensated factor tilts** — beta,
  momentum, residual volatility, liquidity — and **idiosyncratic** alpha, what was earned on
  purpose rather than by accident.
- **Brinson-Fachler again, on the residual**, the third pass: the selection story sharpens, and
  it says whether the Sharpe survives once the factor turns.

**What it settles and what it does not** is in [`../../AGENTS.md`](../../AGENTS.md) — including the
four counterfactual books that answer what the factor model cannot.

**Two inputs are supplied by hand**, from `Data/Curator/Benchmarks/` and `Data/Curator/Factors/` —
the benchmark's weights and returns, and the factor returns. No price provider sells them. Getting
their layout wrong makes the loader read the attribution transposed rather than fail, so shape them
in one place and say what is missing before trying.

**The book arrives daily.** The attribution library rejects a weight file that is not a daily
series, so it reads the book as the engine held it each trading day, drift included, from
`Backtest/` — never `portfolio_weights.csv`, which holds only the rebalance dates.

**What binds the window.** The attribution period is the intersection of the factor files and the
benchmark holdings, so it is usually *shorter* than the backtest. The two sets of numbers describe
different periods and must not be compared directly. Record both windows in `FINDINGS_1.md`.

In [ ]:
# EXAMPLE-ONLY CELL
missing = attribution_analysis.report_missing_inputs()

if len(runs) == 0:
    missing.append("no backtest to attribute: step 5 was skipped")

ATTRIBUTION_READY = len(missing) == 0

if not ATTRIBUTION_READY:
    print("step 6 skipped:", "; ".join(missing))
else:
    import kaxanuk.attribution_analysis.performance_attribution

    headline = runs["the rule"]["result"]
    daily_weights = backtest_engine.read_daily_weights(headline)
    benchmark_weights = attribution_analysis.load_benchmark_weights(daily_weights.index)
    # The benchmark is compared whole: every constituent the book does not hold enters at zero
    # weight, each with its own price series. A book naming only what it holds is compared against
    # the fraction of the index it happens to own, and the difference comes back as alpha.
    book = attribution_analysis.widen_to_benchmark(
        daily_weights,
        benchmark_weights,
        backtest_engine.BENCHMARK_IDENTIFIER,
    )
    benchmark = benchmark_weights.reindex(columns=book.columns).fillna(0.0)
    # A blocking row in the register applies to every stage, not only to the rule: one of them,
    # PARA, contributed 160 percentage points to the index's reconstructed return in a single day.
    book = book.drop(columns=list(EXCLUDED_IDENTIFIERS), errors="ignore")
    benchmark = benchmark.drop(columns=list(EXCLUDED_IDENTIFIERS), errors="ignore")
    benchmark = benchmark.div(benchmark.sum(axis=1), axis=0).fillna(0.0)
    universe = tuple(book.columns)
    asset_returns = attribution_analysis.load_asset_returns(universe, book.index)
    factor_returns = attribution_analysis.load_factor_returns()

    factor_dates = factor_returns["f_market"].index
    window = book.index[(book.index >= factor_dates.min()) & (book.index <= factor_dates.max())]
    print(f"backtest window:    {book.index.min().date()} to {book.index.max().date()}")
    print(f"attribution window: {window.min().date()} to {window.max().date()}")
    print(f"securities priced: {asset_returns.shape[1]} of {len(universe)} the two books name")
    print(f"factor files: {len(factor_returns)}, {', '.join(sorted(factor_returns))}")

In [ ]:
# EXAMPLE-ONLY CELL
# First cut: Brinson-Fachler. Active return into allocation, selection and interaction.
if ATTRIBUTION_READY:
    brinson = kaxanuk.attribution_analysis.performance_attribution.BrinstonFachlerArrowAttribution(
        attribution_analysis.to_arrow(asset_returns.loc[window]),
        attribution_analysis.to_arrow(book.loc[window]),
        attribution_analysis.to_arrow(benchmark.loc[window]),
        date_column=attribution_analysis.DATE_HEADER,
    )
    # The methods compute in place and return nothing: the tables live on the object.
    brinson.time_series_calculation()
    brinson_daily = brinson.df.to_pandas().set_index("date")
    brinson_totals = brinson_daily[["alpha", "allocation", "selection", "interaction"]].sum() * 100
    print("Brinson-Fachler, summed over the window, in percentage points:")
    print(brinson_totals.round(2).to_string())
    print()
    print("This library's first cut is per asset, not per group: each line is the weighting of")
    print("names, not of sectors. The group story is the third pass, on the residual.")

In [ ]:
# EXAMPLE-ONLY CELL
# Second layer: the factor model. Excess return into compensated tilts and idiosyncratic alpha.
if ATTRIBUTION_READY:
    by_factor = {}

    for name, frame in factor_returns.items():
        by_factor[name] = attribution_analysis.to_arrow(frame.loc[window.min():window.max()])
    factor_model = kaxanuk.attribution_analysis.performance_attribution.KNFMArrowAttribution(
        attribution_analysis.to_arrow(book.loc[window]),
        by_factor,
        attribution_analysis.to_arrow(asset_returns.loc[window]),
        date_column=attribution_analysis.DATE_HEADER,
    )
    factor_model.multifactor_attribution()
    factor_daily = factor_model.portfolio_attribution_ts.to_pandas()
    factor_totals = factor_daily.select_dtypes("number").sum() * 100
    reserved_names = attribution_analysis.RESERVED_FACTOR_NAMES
    reserved = [name for name in factor_totals.index if name in reserved_names]
    priced = [name for name in factor_totals.index if name not in reserved_names]
    print("Factor model, summed over the window, in percentage points:")
    print(factor_totals[priced].round(2).sort_values(ascending=False).to_string())
    print()
    print("The reserved series, which are totals rather than factors:")
    print(factor_totals[reserved].round(2).to_string())

## 6 · Counterfactuals — who earned the idiosyncratic share

The factor model leaves part of the book's excess return unexplained. That is a number, not an
answer: the book makes choices the index does not. **Each counterfactual removes exactly one of
those and keeps the rest**, which is the only way the question stops being an inference. The
engine can already price all of them; `AGENTS.md`, under *What attribution must report*, names four
follow-ups.

<!-- example: begin -->

Here the factor model leaves 36.19 points of the rule's 160.02 points of excess return
unexplained, and the book makes choices the index does not — it holds twenty names rather than
about six hundred, picks them by traded value, requires the cross, sells a name the day after its
cross breaks, and weights them equally. Three arms are priced against it: the control removes the
cross, the diagnostic arm the fast exit, and five random books the ranking and the cross together.
Positions equalised within a date are the rule itself, and shifted dates are the delay cells of
section 4's perturbation.

<!-- example: end -->

In [ ]:
# EXAMPLE-ONLY CELL
# The counterfactual books, and the reason for each, written before any of them was priced.
#
#   the control         already priced: the same names without the cross, on the rule's dates.
#                       Against the rule it prices the cross, and nothing else.
#   the diagnostic arm  already priced: the rule with the first design's band. Against the rule
#                       it prices exit speed.
#   the random books    twenty names drawn from the index's members with a price, equal weight, on
#                       the rule's first-of-month dates and held between them. It removes the
#                       ranking and the cross and keeps the size and the calendar. One draw is one
#                       sample, so five seeds, and the spread is the result.
#
# None of these is a candidate. They are diagnostics, and they are in the trial count.
#
# Noted after the run, 2026-09-24: the random books' dates are the first trading days of each
# month in the window, not the rule's. The first, the window's first day, is not a trade date of
# the rule, and the lagged pool is empty on it, so each random book holds only cash until its
# second draw.
RANDOM_SEEDS = (11, 22, 33, 44, 55)
in_point_in_time = (mark.index >= POINT_IN_TIME_START) & (mark.index <= WINDOW_END)
pool = eligible_matrix(False, AVERAGES)
lagged_pool = portfolio_construction.lag_eligibility(pool.loc[in_point_in_time])
random_weights = {}

for seed in RANDOM_SEEDS:
    rows_by_date = {}

    for position, date in enumerate(REEQUALISE_DATES):
        available = lagged_pool.loc[date]
        names = available[available].index.to_series()
        draw_size = min(BOOK_SIZE, len(names))
        row = pandas.Series(0.0, index=lagged_pool.columns)

        if draw_size > 0:
            drawn = names.sample(n=draw_size, random_state=seed * 1000 + position)
            row[drawn.index] = 1.0 / BOOK_SIZE

        rows_by_date[date] = row

    random_weights[seed] = pandas.DataFrame(rows_by_date).transpose()

depth = lagged_pool.loc[REEQUALISE_DATES].sum(axis=1)
print(f"the pool the random books draw from: {depth.min()} to {depth.max()} names per date")

In [ ]:
# EXAMPLE-ONLY CELL
# Priced by the same engine, on the same window, at the same costs. Nothing here may differ from
# the headline run except the book itself.
if backtest_engine.ENGINE_INSTALLED:
    counterfactuals = {}

    for seed in RANDOM_SEEDS:
        counterfactuals[f"random {seed}"] = random_weights[seed]

    for label, weights in counterfactuals.items():
        price(label, weights, POINT_IN_TIME_START, WINDOW_END)

In [ ]:
# EXAMPLE-ONLY CELL
# The factor model on every arm, treated exactly as the headline book was: widened to the whole
# benchmark, blocking names dropped, the same factor files over the same window.


def decompose(priced):
    """
    One book's excess return split into compensated factor exposure and what is left.

    The treatment is the headline book's, so the arms are comparable with it and with each other.
    """
    weights = backtest_engine.read_daily_weights(priced)
    widened = attribution_analysis.widen_to_benchmark(
        weights,
        benchmark_weights,
        backtest_engine.BENCHMARK_IDENTIFIER,
    )
    cleaned = widened.drop(columns=list(EXCLUDED_IDENTIFIERS), errors="ignore")
    aligned = cleaned.reindex(columns=asset_returns.columns).fillna(0.0)
    unpriced = cleaned.drop(columns=asset_returns.columns, errors="ignore")
    model = kaxanuk.attribution_analysis.performance_attribution.KNFMArrowAttribution(
        attribution_analysis.to_arrow(aligned.loc[window]),
        by_factor,
        attribution_analysis.to_arrow(asset_returns.loc[window]),
        date_column=attribution_analysis.DATE_HEADER,
    )
    model.multifactor_attribution()
    daily = model.portfolio_attribution_ts.to_pandas()
    totals = daily.select_dtypes("number").sum() * 100
    totals["weight unpriced"] = float(unpriced.abs().to_numpy().sum())

    return totals


if ATTRIBUTION_READY:
    arms = {
        "the rule": headline,
        "the control": runs["the control"]["result"],
        "the diagnostic arm": runs["the diagnostic arm"]["result"],
    }

    for label in counterfactuals:
        arms[label] = runs[label]["result"]

    decomposition = pandas.DataFrame({label: decompose(priced) for label, priced in arms.items()})
    interesting = [
        "f_total_excess_returns",
        "f_total_factor_returns",
        "f_idyo_returns",
        "f_market",
        "momentum",
        "beta",
        "weight unpriced",
    ]
    present = [name for name in interesting if name in decomposition.index]
    print("Factor model by arm, percentage points over the attribution window:")
    print(decomposition.loc[present].round(2).to_string())
    print()
    random_share = decomposition.loc["f_idyo_returns", [f"random {seed}" for seed in RANDOM_SEEDS]]
    print(f"idiosyncratic points, the rule: {decomposition.loc['f_idyo_returns', 'the rule']:.1f}")
    print(f"the control: {decomposition.loc['f_idyo_returns', 'the control']:.1f}")
    print(f"random books: {random_share.min():.1f} to {random_share.max():.1f}, "
          f"mean {random_share.mean():.1f}")

## 7 · Verdict

**In words.** A notebook that ends in a number and no sentence gets read as whatever the reader
hoped.

Three sentences: does the book work; what attribution says about why; what the next experiment
should change. Then copy the numbers into [`FINDINGS_1.md`](FINDINGS_1.md) — **that file is the
record, this notebook is the method.**

If sections 4 and 5 reported "not installed", this notebook has produced a book and no result,
which is the honest outcome and not a failure.

## Handoff

| Output | Consumed by |
| --- | --- |
| `Portfolio/portfolio_weights.csv` | the backtest engine |
| `Portfolio/` — the readable book and the summaries | humans, and `FINDINGS_1.md` |
| `Backtest/` — the track record, and the book's daily weights | the attribution library, `FINDINGS_1.md`, and the comparison baseline for every later experiment |
| `Attribution/` | `FINDINGS_1.md`, and the comparison baseline for every later experiment |

Later experiments read the **same** panel, over the same window, with the same costs and the same
rebalancing convention, and change only the selection or the weighting — which is what makes the
comparison against this benchmark meaningful.

## Open items to carry forward

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **No result without the licensed engines.** | Without them the book is built but its performance is not measured, and the blueprint's return predictions stay open. |
| 2 | **Turnover is target-to-target, not realised.** | The realised figure is lower. The engine's own series is the one to quote. |
| 3 | **The attribution window is shorter than the backtest**, bound by the supplied files' coverage. | The two sets of numbers describe different periods. |
| 4 | **Delisting exits use one day of hindsight.** | Inert on a universe of live securities; load-bearing on any universe that retains delisted names. |
| 5 | **Cash is a real instrument**, so going flat costs commission and earns a yield. | A strategy that trades to cash often is partly a bet on the front end of the curve. Ask attribution about it. |
| 6 | **Every lever the benchmark declines** — a weight cap, a minimum holding count, risk-aware sizing — is a later experiment, and each has to beat this book to earn its place. | Complexity is added one lever at a time. |

In [ ]:
# EXAMPLE-ONLY CELL
# The verdict is written in FINDINGS_1.md from what this notebook printed; this cell prints the
# three comparisons the blueprint fixed, from the engine's own figures, and nothing else.
if len(runs) > 0:
    print(f"the rule against the index: Sharpe {rows['the rule']['Sharpe']:.3f} against "
          f"{rows['the index, same window']['Sharpe']:.3f}")
    print(f"the rule against its control: Sharpe {sharpe_margin:+.3f}, "
          f"CAGR {cagr_margin:+.2f} points")
    print(f"the kill switch: the rule ahead on both measures in {SUB_PERIODS_AHEAD} of 3 "
          f"sub-periods; trips: {KILL_SWITCH_TRIPS}")
    print(f"criterion 3: {same_sign} of {len(SWEEP)} cells keep the sign; "
          f"passed: {CRITERION_3_READ_AS_PASSED}")
    print(f"the diagnostic arm reproduces the delay: {ARM_REPRODUCES_DELAY}")
    arm_margin = (rows["the rule"]["CAGR"] - rows["the diagnostic arm"]["CAGR"]) * 100
    print(f"the rule against the diagnostic arm: CAGR {arm_margin:+.2f} points")

## 8 · Verify

**Assertions that raise when this experiment's output is wrong.** Section 2.1 prints its
invariants; this section raises on them, and on what the notebook wrote, so a run that reaches the
last cell is one whose book and numbers hold. At the least:

- every invariant of section 2.1 holds;
- `Portfolio/portfolio_weights.csv`, read back, has one column per rebalance date, each summing to
  one with the cash proxy, and no negative weight;
- **every engine run valued the window it was asked for**: the days it valued, counted against the
  trading days in that window — never against a fixed floor, which a run that stopped years early
  can clear. Nearly every trading day valued, and none missing from the window's end, where a
  truncated run loses them. Skipped, as section 4 is, when the engine is not installed;
- the attribution window lies inside the backtest's, and every arm has an idiosyncratic figure.
  Skipped, as section 5 is, when attribution did not run.

In [ ]:
# EXAMPLE-ONLY CELL
# The book: the invariants section 2.1 printed, raised here, and the weight file read back from disk
# rather than from the frame that wrote it.
written_weights = pandas.read_csv(weight_file, index_col=0)
column_totals = written_weights.sum(axis=0)
book_verifications = {
    **checks,
    "one column per trade date": written_weights.shape[1] == len(TRADE_DATES),
    "every column sums to one, cash included": bool(
        ((column_totals - 1.0).abs() <= 0.0001).all()
    ),
    "the cash proxy has a row": backtest_engine.CASH_IDENTIFIER in written_weights.index,
    "no negative weight": bool((written_weights >= 0.0).all(axis=None)),
    "no weight file reaches past the window": bool(
        pandas.to_datetime(written_weights.columns).max() <= WINDOW_END
    ),
}
failed_book = [
    name
    for name, passed in book_verifications.items()
    if not passed
]

if len(failed_book) > 0:
    message = f"the book failed verification: {'; '.join(failed_book)}"

    raise AssertionError(message)

print(f"verified: {len(book_verifications)} checks on the book and {weight_file.name}")

In [ ]:
# EXAMPLE-ONLY CELL
# Every engine run, counted against the trading days of the window it was asked for. A run that
# stops valuing the book still reports success and summarises the stub; the window's own days are
# the check. A day or two at the edges is the engine's calendar, not a truncation.
VALUED_SHARE_REQUIRED = 0.99
UNVALUED_DAYS_AT_END_ALLOWED = 5
run_verifications = {}

for label, run in runs.items():
    run_days = mark.index[(mark.index >= run["start"]) & (mark.index <= run["end"])]
    valued_days = pandas.DatetimeIndex(backtest_engine.read_daily_weights(run["result"]).index)
    valued_in_window = run_days.intersection(valued_days)
    valued_share = len(valued_in_window) / len(run_days)
    unvalued_at_end = run_days[run_days > valued_days.max()]
    run_verifications[f"{label}: {valued_share:.1%} of the window's trading days valued"] = (
        valued_share >= VALUED_SHARE_REQUIRED
    )
    run_verifications[f"{label}: {len(unvalued_at_end)} trading days unvalued at the end"] = (
        len(unvalued_at_end) <= UNVALUED_DAYS_AT_END_ALLOWED
    )

if len(runs) == 0:
    print("engine checks skipped, as section 4 was: the KaxaNuk Backtest Engine is not installed")

if ATTRIBUTION_READY:
    run_verifications["the attribution window lies inside the backtest's"] = bool(
        len(window) > 0
        and window.min() >= book.index.min()
        and window.max() <= book.index.max()
    )
    run_verifications["every arm has an idiosyncratic figure"] = bool(
        decomposition.loc["f_idyo_returns"].notna().all()
    )

failed_runs = [
    name
    for name, passed in run_verifications.items()
    if not passed
]

if len(failed_runs) > 0:
    message = f"the runs failed verification: {'; '.join(failed_runs)}"

    raise AssertionError(message)

print(f"verified: {len(run_verifications)} checks on {len(runs)} engine runs")